In [ ]:
import glob
import matplotlib.pyplot as plt
from IPython.display import display, Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import numpy as np

In [ ]:
def convert_to_pixels(normalized_coords, image_width, image_height):

    """ Convert normalized coordinates to pixel coordinates. 
    Args:
        normalized_coords (list of tuples): List of tuples containing normalized coordinates (x, y).
        image_width (int): Width of the image in pixels.
        image_height (int): Height of the image in pixels.
    Returns:
        list of tuples: List of tuples containing pixel coordinates (x_pixel, y_pixel). 
    
    """

    pixel_coords = []
    for x_norm, y_norm in normalized_coords:
        x_pixel = int(float(x_norm) * image_width)
        y_pixel = int(float(y_norm) * image_height)
        pixel_coords.append((x_pixel, y_pixel))
    return pixel_coords

def process_file(file_path, image_width, image_height):
    bounding_boxes = []
    with open(file_path, 'r') as file:
        for line in file:
            parts = line.split()[1:]  
            normalized_coords = [(parts[i], parts[i+1]) for i in range(0, len(parts), 2)]
            bounding_boxes.append(convert_to_pixels(normalized_coords, image_width, image_height))
    return bounding_boxes

image_width = 640 
image_height = 640


bounding_boxes = process_file(f'/datasets/prediction_a/predict/labels/patch_0.txt', image_width, image_height)


In [ ]:

def plot_bounding_boxes(image_path, bounding_boxes):

    """ this code just verify if the predicted images and labels are correctly forming the boundary boxes or not"""

    img = Image.open(image_path)
    fig, ax = plt.subplots()
    ax.imshow(img)

    for box in bounding_boxes:
        
        poly = np.array(box)
        poly = np.append(poly, [poly[0]], axis=0)  # close the polygon
        patch = patches.Polygon(poly, linewidth=1, edgecolor='r', facecolor='none')
        ax.add_patch(patch)

    plt.axis('off')
    plt.show()
    
image_path = f'/datasets/patches/output_patches_a/patch_0.jpg' 
plot_bounding_boxes(image_path, bounding_boxes)

In [ ]:
def plot_bounding_boxes(image_path, bounding_boxes):

    """ just verifying if the pixelated coordinates are correct or not!!"""

    img = Image.open(image_path)
    fig, ax = plt.subplots()
    ax.imshow(img)

    for box in bounding_boxes:
        
        poly = np.array(box)
        poly = poly.reshape((-1, 2))  
        poly = np.append(poly, [poly[0]], axis=0)  
        patch = patches.Polygon(poly, linewidth=1, edgecolor='r', facecolor='none')
        ax.add_patch(patch)

    plt.axis('off')
    plt.show()

plot_bounding_boxes(image_path, bounding_boxes)

In [ ]:
def convert_to_bounding_boxes(polygon_coords):

    """ Convert polygon coordinates to bounding boxes. 
    Args:
        polygon_coords (list of lists): List of polygons, where each polygon is a list of tuples (x, y).
    Returns:
        list of tuples: List of bounding boxes, where each bounding box is represented as (min_x, min_y, width, height).   
    
    """


    bounding_boxes = []
    for polygon in polygon_coords:
    
        min_x = min_y = float('inf')
        max_x = max_y = float('-inf')

        for (x, y) in polygon:
            if x < min_x:
                min_x = x
            if x > max_x:
                max_x = x
            if y < min_y:
                min_y = y
            if y > max_y:
                max_y = y
        width = max_x - min_x
        height = max_y - min_y
        bounding_boxes.append((min_x, min_y, width, height))

    return bounding_boxes

def plot_bounding_boxes(image_path, bounding_boxes):
    img = Image.open(image_path)
    fig, ax = plt.subplots()
    ax.imshow(img)

    for (x, y, width, height) in bounding_boxes:
        rect = patches.Rectangle((x, y), width, height, linewidth=1, edgecolor='r', facecolor='none')
        ax.add_patch(rect)

    plt.axis('off')
#     plt.savefig(output_path, bbox_inches='tight', pad_inches=0)  # Save the figure
    plt.show()  


polygon_coords = bounding_boxes 
bd_boxes = convert_to_bounding_boxes(polygon_coords)
print(bd_boxes)
plot_bounding_boxes(image_path, bd_boxes)

In [ ]:
def translate_patch_coordinates_to_resized(detected_box, patch_box, resized_size, patch_size):

    """ coordinates of the detected bounding boxes on crypts based on the resized image."""

    px, py, pwidth, pheight = detected_box

    x1, y1, x2, y2 = patch_box
    resized_x = x1 + px
    resized_y = y1 + py
    resized_width = pwidth
    resized_height = pheight

    return (resized_x, resized_y, resized_width, resized_height)


detected_boxes =  bd_boxes 

patch_box = (1280, 640, 1920, 1280) 

resized_boxes = [translate_patch_coordinates_to_resized(box, patch_box, resized_size=(2000, 1979), patch_size=640) for box in detected_boxes]


for i, original_coordinates in enumerate(resized_boxes):
    print(f"Coordinates of detected object {i+1} on resized image:", original_coordinates)

In [ ]:
def translate_patch_coordinates_to_resized(detected_box, patch_box, resized_size, patch_size):
    px, py, pwidth, pheight = detected_box

    x1, y1, x2, y2 = patch_box

    resized_x = x1 + px
    resized_y = y1 + py
    resized_width = pwidth
    resized_height = pheight

    return (resized_x, resized_y, resized_width, resized_height)


detected_box = (44, 268, 38, 116)  
patch_box = (640, 0, 1280, 640) 
original_coordinates = translate_patch_coordinates_to_resized(detected_box, patch_box, resized_size=(2000, 1979), patch_size=640)
print("Coordinates of detected object on resized image:", original_coordinates)

In [ ]:
def scale_patch_coordinates_to_original(detected_box, patch_box, scale_x, scale_y):

    """ coordinates of the detected bounding boxes on crypts based on the original image (47k*48k) """


    px, py, pwidth, pheight = detected_box

    x1, y1, x2, y2 = patch_box

    orig_x = int(x1 + px * scale_x)
    orig_y = int(y1 + py * scale_y)
    orig_width = int(pwidth * scale_x)
    orig_height = int(pheight * scale_y)

    return (orig_x, orig_y, orig_width, orig_height)

detected_box = (84, 190, 94, 182)  
patch_box = (15237, 0, 30474, 15564)  
scale_x = 47000 / 2000 
scale_y = 48000 / 1979  


original_coordinates = scale_patch_coordinates_to_original(detected_box, patch_box, scale_x, scale_y)
print("Original coordinates of detected object:", original_coordinates)